In [ ]:
# ==== THAM SỐ (cell duy nhất cần sửa) — ánh xạ 1-1 tới CLI của preprocess_* / make_splits / pack_h5 ====
REPO_URL = "https://github.com/NguyenMinhTri24072005/MV_CAAFNet.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/MV_CAAFNet"

DATASET = "luna16"                                                      # "luna16" | "lidc"
DATA_CFG = "configs/data.yaml"                                          # --config
RAW_LIDC_DIR = f"{DRIVE_ROOT}/data/raw/LIDC-IDRI"                       # --lidc-dir / --raw-dir
RAW_LUNA16_DIR = f"{DRIVE_ROOT}/data/raw/LUNA16"                        # --raw-dir (khi luna16)
INDEX_DIR = f"{DRIVE_ROOT}/data/index"                                  # --index-dir
INTERIM_DIR = f"{DRIVE_ROOT}/data/interim/{DATASET}"                    # --out-dir (có _DONE để resume nếu ngắt phiên)
SPLITS_OUT = f"{DRIVE_ROOT}/data/splits/{DATASET}.json"                 # --out (make_splits)
LOCAL_PACK_TMP = f"/content/processed_tmp/{DATASET}"                    # --out-dir (pack_h5 ghi SSD cục bộ trước)
DRIVE_PROCESSED_OUT = f"{DRIVE_ROOT}/data/processed/{DATASET}"          # --copy-to-drive-dir (đổi thành processed_s1_slice khi làm S1)

SPLIT_MODE = None                                                       # --mode (None = theo data.yaml | "patient" | "slice")
WORKERS = 4                                                             # --workers (2-4 tối ưu Drive FUSE)
LIMIT_SCANS = None                                                      # --limit (None = toàn bộ; ví dụ 5 để chạy thử nhanh)
RUN_PREPROCESS = True
RUN_SPLIT = True
RUN_PACK = True

# 01 — Tiền xử lý, chia tập & đóng gói HDF5 (Phase 5)

**Runtime:** Colab Pro **CPU High-RAM**. Chạy riêng cho từng bộ dữ liệu (`DATASET = "luna16"` rồi `"lidc"`).

- Chỉ sửa **cell tham số**. Cần có `data/index/` từ `00_colab_data_survey.ipynb`.
- Tiền xử lý đọc thẳng từ Drive và **resume theo `_DONE`**: nếu Colab ngắt phiên, chạy lại cell là tiếp tục.
- `pack_h5` ghi `.h5` ra SSD `/content/processed_tmp/` trước, rồi chép **một lần** sang Drive và kiểm SHA-256.
- Thử nghiệm S1 (Phase 8b): đặt `SPLIT_MODE = "slice"`, đổi `SPLITS_OUT` sang thư mục khác (giữ tên tệp `<ds>.json`)
  và `DRIVE_PROCESSED_OUT` sang `data/processed_s1_slice/<ds>`.

In [ ]:
# Setup chung: mount Drive → clone/pull repo (GH_TOKEN qua GIT_ASKPASS, không in token) → cài thư viện
import os, platform, shlex, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/MV_CAAFNet"


def _git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")          # Colab Secrets → không ghi token vào notebook
    except Exception:
        token = None                               # repo public hoặc chưa cấu hình secret
    if token:
        askpass = Path("/content/.git_askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GH_TOKEN=token)
    return env


def _git(*args):
    subprocess.run(["git", *args], check=True, env=_git_env())


if Path(REPO_DIR, ".git").is_dir():
    _git("-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH)
    _git("-C", REPO_DIR, "checkout", "--quiet", BRANCH)
    _git("-C", REPO_DIR, "pull", "--quiet", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)


def run(args):
    """Chạy `python <args>` từ thư mục repo (danh sách tham số → an toàn với đường dẫn có dấu cách)."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024 ** 3
print(f"commit {commit} | Python {platform.python_version()} | {os.cpu_count()} vCPU | RAM {mem_gb:.1f} GB")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    print("PyTorch: không có (không cần cho notebook CPU này)")

## 1. Tiền xử lý (resume theo `_DONE`)

In [ ]:
import json

import pandas as pd
from IPython.display import display

assert DATASET in ("luna16", "lidc"), DATASET
if RUN_PREPROCESS:
    if DATASET == "lidc":
        args = ["-m", "src.data.preprocess_lidc", "--raw-dir", RAW_LIDC_DIR]
    else:
        args = ["-m", "src.data.preprocess_luna16", "--raw-dir", RAW_LUNA16_DIR, "--lidc-dir", RAW_LIDC_DIR]
    args += ["--index-dir", INDEX_DIR, "--out-dir", INTERIM_DIR, "--config", DATA_CFG, "--workers", WORKERS]
    if LIMIT_SCANS:
        args += ["--limit", LIMIT_SCANS]
    run(args)

interim = Path(INTERIM_DIR)
summary = json.loads((interim / "preprocess_summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in summary.items() if k != "data_config"}, ensure_ascii=False, indent=2))
errors = pd.read_csv(interim / "errors.csv")
print(f"errors.csv: {len(errors)} scan lỗi (chạy lại cell để thử lại các scan này)")
display(errors.head(50))
if DATASET == "luna16":
    match = pd.read_csv(interim / "match_report.csv")
    display(match["status"].value_counts().rename("số nốt").to_frame())
patches = pd.read_csv(interim / "patches.csv", dtype={"patch_id": str, "patient_id": str})
print(f"{len(patches)} patch, {patches['patient_id'].nunique()} bệnh nhân, {patches['series_uid'].nunique()} scan")
display(patches.describe().T[["mean", "std", "min", "max"]])

## 2. Chia tập (`patient`: đúng 1.200 test, 0% rò rỉ bệnh nhân)

In [ ]:
if RUN_SPLIT:
    args = ["-m", "src.data.make_splits", "--index", f"{INTERIM_DIR}/patches.csv", "--out", SPLITS_OUT,
            "--config", DATA_CFG]
    if SPLIT_MODE:
        args += ["--mode", SPLIT_MODE]
    run(args)
splits = json.loads(Path(SPLITS_OUT).read_text(encoding="utf-8"))
print("mode:", splits["mode"], "| seed:", splits["seed"], "| bệnh nhân ranh giới:", splits["boundary_patient"])
display(pd.DataFrame({k: v for k, v in splits["stats"].items() if k != "patient_overlap"}).T)
print("Rò rỉ bệnh nhân giữa các tập:", splits["stats"]["patient_overlap"])

## 3. Đóng gói `.h5` trên SSD rồi chép một lần sang Drive (kiểm SHA-256)

In [ ]:
from src.data.pack_h5 import verify_manifest

if RUN_PACK:
    run(["-m", "src.data.pack_h5", "--interim-dir", INTERIM_DIR, "--splits", SPLITS_OUT, "--out-dir", LOCAL_PACK_TMP,
         "--config", DATA_CFG, "--copy-to-drive-dir", DRIVE_PROCESSED_OUT])
bad = verify_manifest(DRIVE_PROCESSED_OUT)
print("SHA-256 trên Drive:", "OK" if not bad else bad)
manifest = json.loads(Path(DRIVE_PROCESSED_OUT, "manifest.json").read_text(encoding="utf-8"))
display(pd.DataFrame(manifest["files"]).T)

## 4. Kiểm tra trực quan: lưới 32 patch (mask khớp nốt, `dist == 0` trong mask)

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

from src.data.compute_edt import edt_from_mask

src_dir = Path(LOCAL_PACK_TMP) if Path(LOCAL_PACK_TMP, "train.h5").exists() else Path(DRIVE_PROCESSED_OUT)
for split in ("train", "val", "test"):
    with h5py.File(src_dir / f"{split}.h5", "r") as h5:
        m, d = h5["mask"][...], h5["dist"][...]
        print(f"{split}: {len(m)} patch | dist == 0 trong mask: {bool((d[m > 0] == 0).all())} | "
              f"tỉ lệ mask rỗng: {(m.reshape(len(m), -1).sum(1) == 0).mean():.2%}")
with h5py.File(src_dir / "train.h5", "r") as h5:
    n = h5["image"].shape[0]
    sel = np.sort(np.random.default_rng(0).choice(n, size=min(32, n), replace=False))
    img, msk, dist = h5["image"][sel], h5["mask"][sel], h5["dist"][sel]
    ids = [x.decode() if isinstance(x, bytes) else x for x in h5["patch_id"][sel]]
assert all(np.allclose(dist[i, 0], edt_from_mask(msk[i, 0])) for i in range(len(sel))), "dist ≠ EDT(mask)"
rows = int(np.ceil(len(sel) / 8))
fig, axes = plt.subplots(rows, 8, figsize=(16, 2.1 * rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, im, mk, pid in zip(axes.ravel(), img, msk, ids):
    ax.imshow(im[0], cmap="gray", vmin=0, vmax=1)
    if mk[0].any():
        ax.contour(mk[0], levels=[0.5], colors="r", linewidths=0.8)
    ax.set_title(pid.replace("LIDC-IDRI-", ""), fontsize=7)
plt.tight_layout()
plt.show()

## Tiếp theo

- Chạy lại notebook với `DATASET = "lidc"` nếu cần bộ LIDC-IDRI.
- `10_colab_train.ipynb` (GPU) sẽ copy `DRIVE_ROOT/data/processed/<ds>/` về `/content/processed` và kiểm SHA-256 trước khi train.